In [1]:
import pymupdf4llm
import re
import os
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_chroma import Chroma
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
file_path = "Attention.pdf"
markdown = pymupdf4llm.to_markdown(file_path)

rapidocr_api using backend: rapidocr

=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.


In [3]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4")
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
markdown_chunks = splitter.split_text(markdown)

In [4]:
def additional_recursive_chunker(markdown_chunks : list[Document], chunk_size = 600, chunk_overlap = 100):

    good_chunk = []
    chunk_count = 0
    recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, 
                                                             chunk_overlap = chunk_overlap, 
                                                             separators=["\n\n", "\n", ". ", " ", ""])
    
    for chunk in markdown_chunks:
        if(len(chunk.page_content) > chunk_size):
            meta_data = chunk.metadata
            chunks = recursive_text_splitter.split_documents([chunk])
            for i in chunks:
                chunk_count = chunk_count + 1
                i.metadata["chunk_id"] = chunk_count
                good_chunk.append(i)
            chunk_count = 0
        else:
            good_chunk.append(chunk)
    return good_chunk

In [5]:
# this is the final chunk and metadata which is to be stores and embedded
final_chunks = additional_recursive_chunker(markdown_chunks)

In [6]:
embedding_model = OllamaEmbeddings(model="llama3.2")

In [7]:
# embed each of the query and then store them in the pgvector
vector_store = Chroma(
    collection_name= "my_docs",
    embedding_function=embedding_model,
    persist_directory="./store"
)

In [ ]:
ids = vector_store.add_documents(
    documents = final_chunks,)

In [ ]:
len(ids)

83

In [8]:
filter_condition = {}
search_kwargs = {}

In [9]:
if len(filter_condition) == 0:
    search_kwargs = {
        'k' : 3,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
    }
else:
    search_kwargs = {
        'k' : 3,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
        "filter" : {
            **filter_condition
        }
    },

In [10]:
search_kwargs

{'k': 3, 'fetch_k': 10, 'lambda_mult': 0.7}

In [11]:
retriver = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = search_kwargs
)

In [12]:
ans =retriver.invoke("How does an attention mechanism work?")

In [13]:
ans

[Document(id='35e55b56-3c2d-4fb3-aded-3ecb083fe0c5', metadata={'chunk_id': 7, 'Header 2': '**1 Introduction**', 'Header 1': '**Attention Is All You Need**'}, page_content='Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network.'),
 Document(id='28bcbc81-a251-4209-a789-b6de8865539c', metadata={'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**', 'chunk_id': 2, 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'Header 3': 'Ouput Probabilities'}, page_content='- The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each

In [14]:
for i in ans:
    print(i.page_content)
    print("=" * 150)

Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network.
- The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.


In [15]:
for i in ans:
    print(i.metadata)

{'chunk_id': 7, 'Header 2': '**1 Introduction**', 'Header 1': '**Attention Is All You Need**'}
{'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**', 'chunk_id': 2, 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'Header 3': 'Ouput Probabilities'}
{'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**', 'Header 3': 'Ouput Probabilities', 'Header 4': '**3.2 Attention**'}


### Evaluation of the rag system without hybrid search and reranking

#### Evaluation LLMs

In [16]:
from openai import OpenAI
from groq import Groq
from ragas.llms import llm_factory

OllamaClient = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434"
)

GroqClient = Groq(
    api_key=os.getenv("api_key")
)

GroqEvaluation = llm_factory(
    model="llama-3.3-70b-versatile",
    client=GroqClient
)

OllamaEvaluation = llm_factory(
    "ollama",
    provider="openai",
    client=OllamaClient
)

c:\Users\Nitro\Desktop\productionReadyRAGapp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Nitro\Desktop\productionReadyRAGapp\venv\Lib\site-packages\ragas\llms\base.py:556: UserWarning: Client should be an instance of openai.OpenAI or openai.AsyncOpenAI. Unexpected behavior may occur with other client types.
  return instructor.from_openai(client, mode=instructor.Mode.JSON)


#### RAG LLMs

In [17]:
OllamaLLM = ChatOllama(model="llama3.2")
GroqLLM = ChatGroq(
    api_key=os.getenv("api_key"),
    model="llama-3.3-70b-versatile"
)

#### Metrics

In [18]:
from ragas.metrics.collections import ( 
    ContextPrecision,
    ContextRecall, 
    ContextRelevance, 
    Faithfulness, 
    AnswerCorrectness, 
    AnswerRelevancy )
from ragas import evaluate
from datasets import Dataset

In [28]:
faithulness = Faithfulness(llm = OllamaEvaluation)
contextRecall = ContextRecall(llm = OllamaEvaluation)
contextPrecision = ContextPrecision(llm = OllamaEvaluation)
contextRelevance = ContextRelevance(llm = OllamaEvaluation)
answerCorrectness = AnswerCorrectness(llm = OllamaEvaluation, embeddings = embedding_model)
# answerRelevancy = AnswerRelevancy(llm = OllamaEvaluation, embeddings= embedding_model) 

In [19]:
# evaluation retriever
def recall_k(total_relevant_chunks, total_relevant_chunks_retirevied):
    # -> how many of relevant chunks were retreived out of all the relevant chunks available
    if total_relevant_chunks == 0:
        return 0.0
    recall = total_relevant_chunks_retirevied/total_relevant_chunks
    return recall

def precision_k(total_retreived_chunks, total_relevant_ones):
    # -> how many of the retrieved chunks were relevant chunks to the the question
    if total_retreived_chunks == 0:
        return 0.0
    precision = total_relevant_ones/total_retreived_chunks
    return precision

# evaluate the ranker (reranker or retriever's ranker)
def mrr(llm):
    # -> the first relevant chunk out of all the retrieved chunk which are returned in order
    pass

def ndcg(llm):
    # -> this metrics check whether the chunks are arranged in an descending order of chunks on 
    # the basis of relevancy
    pass

#### dataset for evaluation
the golden dataset will contain the dataset which will contain:
- user_query
- response
- retreived_contexts
- reference 

In [30]:
dataset = [
    {
        "query": "What is the Transformer architecture?",
        "ground_truth": "The Transformer is a neural network architecture based entirely on attention mechanisms, replacing recurrence and convolutions.",
        "expected_answer": "The Transformer uses attention mechanisms instead of recurrent or convolutional layers.",
        "citations": [
            {
                "page": 1,
                "section": "Abstract",
                "reference_text": "The Transformer is based solely on attention mechanisms, dispensing with recurrence and convolutions entirely."
            }
        ]
    },
    {
        "query": "What is self-attention and how does it work?",
        "ground_truth": "Self-attention relates different positions of a single sequence to compute a representation of that sequence.",
        "expected_answer": "Self-attention allows each token to attend to other tokens in the same sequence.",
        "citations": [
            {
                "page": 3,
                "section": "3.2.3 Applications of Attention in our Model",
                "reference_text": "In 'self-attention' the queries, keys and values come from the same place, in this case, the output of the previous layer."
            }
        ]
    },
    {
        "query": "What are Query, Key, and Value in the attention mechanism?",
        "ground_truth": "Queries and keys are used to calculate attention scores, while values are weighted to produce the attention output.",
        "expected_answer": "Queries represent what to look for, keys determine relevance, and values provide the information to aggregate.",
        "citations": [
            {
                "page": 3,
                "section": "3.2 Attention",
                "reference_text": "An attention function can be described as mapping a query and a set of key-value pairs to an output."
            }
        ]
    },
    {
        "query": "Why does the Transformer use multi-head attention?",
        "ground_truth": "Multiple attention heads allow the model to jointly attend to information from different representation subspaces at different positions.",
        "expected_answer": "Multiple heads allow the model to capture different relationships and information simultaneously.",
        "citations": [
            {
                "page": 4,
                "section": "3.2.2 Multi-Head Attention",
                "reference_text": "This allows the model to jointly attend to information from different representation subspaces at different positions."
            }
        ]
    },
    {
        "query": "What is positional encoding and why is it necessary?",
        "ground_truth": "Positional encoding injects information about token positions into embeddings because the Transformer does not use recurrence or convolution.",
        "expected_answer": "Positional encoding gives the Transformer information about the order of tokens in a sequence.",
        "citations": [
            {
                "page": 5,
                "section": "3.5 Positional Encoding",
                "reference_text": "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence."
            }
        ]
    },
    {
        "query": "What is the difference between encoder-decoder attention and masked self-attention?",
        "ground_truth": "Encoder-decoder attention allows decoder positions to attend to encoder outputs, whereas masked self-attention prevents decoder positions from attending to subsequent future positions.",
        "expected_answer": "Encoder-decoder attention retrieves information from the input sequence through encoder outputs, while masked self-attention prevents the decoder from accessing future tokens.",
        "citations": [
            {
                "page": 5,
                "section": "3.1 Encoder and Decoder Stacks",
                "reference_text": "The encoder-decoder attention layer allows every position in the decoder to attend over all positions in the input sequence."
            },
            {
                "page": 5,
                "section": "3.1 Encoder and Decoder Stacks",
                "reference_text": "We modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions."
            }
        ]
    }
]

In [ ]:
# this will contain the dataset with keys:

golden_dataset = []

# retrieve from the document
for data in dataset[::-1]:
    question = data['query']
    print(question)
    retrieved_chunks = retriver.invoke(question)
    print(retrieved_chunks)

    print("="*150)
    citation = [citation['section'] for citation in data['citations']]
    for i in range(len(retrieved_chunks)):
        print(retrieved_chunks[i].metadata)
    break


What is the difference between encoder-decoder attention and masked self-attention?
[Document(id='28bcbc81-a251-4209-a789-b6de8865539c', metadata={'Header 1': '**Attention Is All You Need**', 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'chunk_id': 2, 'Header 2': '**3 Model Architecture**', 'Header 3': 'Ouput Probabilities'}, page_content='- The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.'), Document(id='e409ea17-a148-4011-a536-2af65d17163b', metadata={'Header 2': '**2 Background**', 'Header 1': '**Attention Is All You Need**', 'chunk_id': 1}, page_content='The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural netw

In [24]:
eval_dataset = Dataset.from_list(golden_dataset)
eval_dataset

Dataset({
    features: [],
    num_rows: 0
})

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

### RAG with hybrid search

In [22]:
# we need to perform the retrieval on the basis of the semantic and the keyword and the fuse then perform rrf on them to find the best chunk
# semantic search is the normal search for this


### BM25 as the keyword search

In [17]:
# keyword search is done using the bm25 algorithm

### Combining the chunks from dense and sparse search

In [18]:
# rrf

In [19]:
# reranker

### Generation of the answer from the llm

In [20]:
# generation

### Evaluate the chunks and answer after the changes in the pipeline

In [21]:
# evaluation criterias